In [1]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from cloelib.cosmology.camb_cosmology import CAMBBackground
from cloelib.observables.CometEFT_spectro import CometEFT_SpectroPower
from cloelike.EuclidLikelihood_GCSpectro_Pks import EuclidLikelihoodGCspectroPks

Warning! "libgrid.so" not found, bispectrum binning options will not be available.


In [2]:
redshifts = [1.0, 1.2, 1.4, 1.65]
labels = [str(z).strip('0') for z in redshifts]

data = {
    'GCspectro': {},
    'fiducial_cosmology': {
        'H0': 67.0,
        'Omega_cdm0': 0.27,
        'Omega_b0': 0.049,
        'Omega_k0': 0.0,
        'w0': -1.0,
        'wa': 0.0,
        'ns': 0.96,
        'As': 2.1e-9,
        'gamma_MG': 0.545,
        'mnu': 0.0
    }
}

fid_h = data['fiducial_cosmology']['H0'] / 100.0

In [3]:
def get_index(multipole, scale, scale_dict):
    offset = sum(len(scale_dict[ell]) for ell in multipoles if ell < multipole)
    return offset + np.where(scale_dict[multipole] == scale)[0][0]

k_fac = fid_h
pk_fac = 1.0 / fid_h**3
cov_fac = 1.0 / fid_h**6

nbar = np.array([2.042611E-03, 1.02876011E-03, 0.58531983E-03, 0.313402E-03]) * fid_h**3

for ii,z in enumerate(labels):

    data['GCspectro'][z] = {}
    
    filename = f'/Users/pezzotta/work/cloe/cloelike/data/cov_EFT_z{z}.fits'
    hdul = fits.open(filename)
    average = hdul['AVERAGE'].data
    data['GCspectro'][z]['k'] = average['SCALE_1DIM'] * k_fac
    data['GCspectro'][z]['pk0'] = average['AVERAGE0'] * pk_fac
    data['GCspectro'][z]['pk2'] = average['AVERAGE2'] * pk_fac
    data['GCspectro'][z]['pk4'] = average['AVERAGE4'] * pk_fac

    
    covariance = hdul['COVARIANCE'].data
    scale_i = covariance["SCALE_1DIM-I"]
    multipole_i = covariance["MULTIPOLE-I"]
    scale_j = covariance["SCALE_1DIM-J"]
    multipole_j = covariance["MULTIPOLE-J"]
    covariance = covariance["COVARIANCE"]

    multipoles = np.array([0, 2, 4])
    scale_dict = {ell: np.unique(scale_i[multipole_i == ell]) for ell in multipoles}
    matrix_size = sum(len(scale_dict[ell]) for ell in multipoles)
    cov_matrix = np.zeros((matrix_size, matrix_size))
    for s_i, m_i, s_j, m_j, cov in zip(scale_i, multipole_i, scale_j, multipole_j, covariance):
        i = get_index(m_i, s_i, scale_dict)
        j = get_index(m_j, s_j, scale_dict)
        cov_matrix[i, j] = cov
    data['GCspectro'][z]['cov'] = cov_matrix * cov_fac

    data['GCspectro'][z]['nbar'] = nbar[ii]

In [4]:
settings = {
    'scale_cuts': {
        'GCspectro': {
            'bin1': {
                'ell0': [0.0, 0.20], 'ell2': [0.0, 0.15], 'ell4': [0.0, 0.15]
            },
            'bin2': {
                'ell0': [0.0, 0.25], 'ell2': [0.0, 0.20], 'ell4': [0.0, 0.20]
            },
            'bin3': {
                'ell0': [0.0, 0.25], 'ell2': [0.0, 0.20], 'ell4': [0.0, 0.20]
            },
            'bin4': {
                'ell0': [0.0, 0.30], 'ell2': [0.0, 0.25], 'ell4': [0.0, 0.25]
            }
        }
    }
}

settings = {
    'scale_cuts': {
        'GCspectro': {
            bin_key: {
                ell_key: [v * fid_h for v in values]
                for ell_key, values in bin_values.items()
            }
            for bin_key, bin_values in settings['scale_cuts']['GCspectro'].items()
        }
    }
}

In [5]:
like = EuclidLikelihoodGCspectroPks(data=data, settings=settings, Background=CAMBBackground, SpectroPower=CometEFT_SpectroPower)

In [6]:
parameters = {
    'H0': 67.0,
    'Omega_cdm0': 0.27,
    'Omega_b0': 0.049,
    'Omega_k0': 0.0,
    'w0': -1.0,
    'wa': 0.0,
    'ns': 0.96,
    'As': 2.1e-9,
    'mnu': 0.0,
    'gamma_MG': 0.545,
    'b1': np.array([1.412, 1.769, 2.039, 2.496]),
    'b2': np.array([0.695, 0.870, 1.162, 2.010]),
    'bG2': np.array([-0.156, -0.299, -0.400, -0.555]),
    'bGam3': np.array([0.323, 0.621, 0.827, 1.137]),
    'c0': np.array([30.948, 37.116, 36.738, 53.627]),
    'c2': np.array([46.233, 53.071, 48.626, 60.962]),
    'c4': np.array([10.057, 10.385, 8.643, 8.711]),
    'cnlo': np.array([0.0, 0.0, 0.0, 0.0]),
    'NP0': np.array([1.056, 1.152, 1.144, 1.309]),
    'NP20': np.array([0.0, 0.0, 0.0, 0.0]),
    'NP22': np.array([0.0, 0.0, 0.0, 0.0]),
    'fout': np.array([0.0, 0.0, 0.0, 0.0]),
    'sigmaz': np.array([0.0, 0.0, 0.0, 0.0])
}

-2*like.loglike(parameters)

5.891766452608903e-09

In [7]:
parameters['H0'] = 70.0

-2*like.loglike(parameters)

54986.321757500504

In [8]:
parameters['H0'] = 67.0
parameters['b1'] = np.array([2.0, 2.0, 2.0, 2.0])

-2*like.loglike(parameters)

391101.88771686115